In [0]:
# -------------------------------------------
# Project Configuration
# -------------------------------------------

CATALOG = "retail_demo"
BRONZE_SCHEMA = "bronze"
SILVER_SCHEMA = "silver"

# We no longer now need the volumn path as we will be reading it via the Delta tables

In [0]:
# Reading data from bronze schema customers table, equivalent to saying give me customers_raw delta table from the bronze_schema
customers_df = spark.table(
    f"{CATALOG}.{BRONZE_SCHEMA}.customers_raw"
)

In [0]:
# We will now display it with limit 10
# display(customers_df.limit(10))

# We will now print schema
# customers_df.printSchema()
# Interesting that metadata already have correct data-types this is because we created them in bronze

In [0]:
# function to get profile of the dataset
from pyspark.sql import DataFrame

def profile_table(df: DataFrame, table_name: str, primary_key: str):

    row_count = df.count()
    distinct_keys = df.select(primary_key).distinct().count()

    print("=" * 60)
    print(f"Table Name        : {table_name}")
    print(f"Total Rows        : {row_count:,}")
    print(f"Total Columns     : {len(df.columns)}")
    print(f"Distinct {primary_key}: {distinct_keys:,}")
    print(f"Duplicate Records : {row_count - distinct_keys:,}")
    print("=" * 60)

    print("\nSchema")
    df.printSchema()

    print("\nSummary Statistics")
    df.describe().show(truncate=False)

In [0]:
profile_table(
    customers_df,
    table_name="customers",
    primary_key="customer_id"
)

In [0]:
# Only work with data or columns that we require
# we're creating a new DataFrame (customers_silver) rather than modifying customers_df.
from pyspark.sql.functions import col

customers_silver = customers_df.select(
    col("customer_id"),
    col("customer_unique_id"),
    col("customer_zip_code_prefix"),
    col("customer_city"),
    col("customer_state"),
    col("ingestion_timestamp"),
    col("source_file")
)

In [0]:
# Everything looks fine except the zip_code
from pyspark.sql.functions import col
from pyspark.sql.types import IntegerType

customers_silver = customers_silver.withColumn(
    "customer_zip_code_prefix",
    col("customer_zip_code_prefix").cast(IntegerType())
)

In [0]:
customers_silver.printSchema()

In [0]:
display(customers_silver.limit(10))

In [0]:
# null check
# what is does it selects the table, and c is the pointer, it is used in the for loop to point a column and each time it loops, it checks isNull(), this returns binary Yes or No then it casts to integer and then it sums the total nulls for each column in the table
from pyspark.sql.functions import col, sum

customers_silver.select([
    sum(col(c).isNull().cast("int")).alias(c)
    for c in [
        "customer_id",
        "customer_unique_id",
        "customer_zip_code_prefix",
        "customer_city",
        "customer_state"
    ]
]).show()

In [0]:
# duplicate check
# we get the count of customers_silver and count the no of rows, we substract this with customers_silver table's primary key distinct count and if we get zero that means 0 duplicate records if we get a positive count that means rows are x and distinct values is less than x. This cannot give us a negative value because the distict values cannot be more than count of rows. Finally we print the result.
duplicate_count = (
    customers_silver.count()
    - customers_silver.select("customer_id").distinct().count()
)

print(f"Duplicate customer_id records: {duplicate_count}")

In [0]:
# Trim validation
# We are here only showing the rows where length of col value from customer_city is not equal to trimmed column value of the same customer_city, basically if the value is Mumbai then it checks if the length is same for both cases if not then it will display
from pyspark.sql.functions import length, trim

customers_silver.filter(
    length(col("customer_city")) != length(trim(col("customer_city")))
).show()

In [0]:
# Trim validation
# We are here only showing the rows where length of col value from customer_state is not equal to trimmed column value of the same customer_state, basically if the value is Maharashtra then it checks if the length is same for both cases if not then it will display
from pyspark.sql.functions import length, trim
customers_silver.filter(
    length(col("customer_state")) != length(trim(col("customer_state")))
).show()

In [0]:
# uppercase validation
# We filter the rows for the customer_silvers where we check if we have any rows where customer_state column value is not upper case

from pyspark.sql.functions import upper

customers_silver.filter(
    col("customer_state") != upper(col("customer_state"))
).show()

In [0]:
# customers_silver.printSchema()
# We did a transformation of datatype for zip code and this makes the schema valid, rest all datatype changes are not required.
# Now lets create an expected schema for a table and what actual schema it has, and create a function which takes these two variables and omits the diff
expected_customers_schema = {
    "customer_id": "string",
    "customer_unique_id": "string",
    "customer_zip_code_prefix": "int",
    "customer_city": "string",
    "customer_state": "string",
    "ingestion_timestamp": "timestamp",
    "source_file": "string"
}

In [0]:
from pyspark.sql import Row


def validate_schema(expected_schema, current_schema):
    """
    Compares expected and current schemas.

    Returns a table containing:
    - column_name
    - expected_data_type
    - current_data_type
    - match
    """

    # Convert Spark StructType into a dictionary
    # this is shorter way of writing a for loop which takes one field at a time from the current_schema.fields, gives a name to the field, and gets it data type into string format and finally put them into the dict
    # For every field in the current_schema create an entry where key is filed.name and value is data type of the filed name
    current_schema_dict = {
        field.name: field.dataType.simpleString()
        for field in current_schema.fields
    }

    results = []

    # Check every expected column
    for column_name, expected_type in expected_schema.items():

        current_type = current_schema_dict.get(column_name)

        match = 1 if current_type == expected_type else 0

        results.append(
            Row(
                column_name=column_name,
                expected_data_type=expected_type,
                current_data_type=current_type,
                match=match
            )
        )

    return spark.createDataFrame(results)

In [0]:
# validate schema is the function and we are validating this with our expected and current schema
schema_check = validate_schema(
    expected_schema=expected_customers_schema,
    current_schema=customers_silver.schema
)

display(schema_check)

In [0]:
# Table specific transformations
from pyspark.sql.functions import col, length, trim

# Check customer_city
city_trim_issues = customers_silver.filter(
    length(col("customer_city")) != length(trim(col("customer_city")))
)

print(f"City values requiring trim: {city_trim_issues.count()}")

In [0]:
# same for customer_state
state_trim_issues = customers_silver.filter(
    length(col("customer_state")) != length(trim(col("customer_state")))
)

print(f"State values requiring trim: {state_trim_issues.count()}")

In [0]:
# Now the uppercase validation for customer_state
from pyspark.sql.functions import upper

state_case_issues = customers_silver.filter(
    col("customer_state") != upper(col("customer_state"))
)

print(f"State values not uppercase: {state_case_issues.count()}")

In [0]:
# One last validation on state lenght which should be 2
state_length_issues = customers_silver.filter(
    length(col("customer_state")) != 2
)

print(f"Invalid state lengths: {state_length_issues.count()}")

In [0]:
spark.sql(f"""
CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SILVER_SCHEMA}
""")

In [0]:
(
    customers_silver.write
        .format("delta")
        .mode("overwrite")
        .saveAsTable(f"{CATALOG}.{SILVER_SCHEMA}.customers")
)

In [0]:
customers_final = spark.table(
    f"{CATALOG}.{SILVER_SCHEMA}.customers"
)

In [0]:
display(customers_final.limit(10))

In [0]:
print(f"Bronze rows : {customers_df.count():,}")
print(f"Silver rows : {customers_final.count():,}")

In [0]:
# This gives a head start where we have used the profile function we have given the function name and table name and primary key as input parameters and got the result
#From the results I can see that we have many columns related to time and it is as follows order_purchase is the order placed, then the payment approved as order_approved, then the order getting handed to the courier partner, then delivery date and then estimated date

orders_df = spark.table(
    f"{CATALOG}.{BRONZE_SCHEMA}.orders_raw"
)

profile_table(
    orders_df,
    table_name="orders",
    primary_key="order_id"
)

In [0]:
# Here we are checking the unique values in order_status column just in case we to check if we have some random things in the status column

orders_df.groupBy("order_status") \
    .count() \
    .orderBy("count", ascending=False) \
    .show()

In [0]:
# Here we are checking for each order status what is the count of NULL values, like order status is delivered and what is the order_approved_at null count or order_delivered_carrier_date and so on
# Its like a matrix view where we get a clear look at each order_status, how many null values we have in each of the three buckets which will enhnace our view to find if there is any data inconsistency, like order status is deleivered but the order_deliver_date is NULL

orders_df.groupBy("order_status") \
    .agg(
        sum(col("order_approved_at").isNull().cast("int")).alias("null_approved"),
        sum(col("order_delivered_carrier_date").isNull().cast("int")).alias("null_carrier"),
        sum(col("order_delivered_customer_date").isNull().cast("int")).alias("null_customer")
    ) \
    .orderBy("order_status") \
    .show()

In [0]:
# Sice we found out that there are 14 orders with NULL approved and 2 orders with null_carrier and 8 orders with null_customer and order_status as delivered, we are here trying to dig deeper to see which records they are.
# We filter on two conditions one is order_status as delivered and second is the null conditions

orders_df.filter(
    (col("order_status") == "delivered") &
    (
        col("order_delivered_customer_date").isNull() |
        col("order_delivered_carrier_date").isNull() |
        col("order_approved_at").isNull()
    )
).select(
    "order_id",
    "order_status",
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date"
).show(truncate=False)

In [0]:
# From the above table we are still not sure if we need to keep the data or not, so here we are checking one more logical condition that says like below
# order_purchase_timestamp > order_approved_at > order_delivered_carrier_date > order_delivered_customer_date and estimated can be anything it can be less than actual delievered time or less but should never be greater than purchase_timestamp

from pyspark.sql.functions import to_timestamp, col

orders_check = orders_df.select(
    "*",
    to_timestamp("order_purchase_timestamp").alias("purchase_ts"),
    to_timestamp("order_approved_at").alias("approved_ts"),
    to_timestamp("order_delivered_carrier_date").alias("carrier_ts"),
    to_timestamp("order_delivered_customer_date").alias("customer_ts"),
    to_timestamp("order_estimated_delivery_date").alias("estimated_ts")
)

orders_check.filter(
    (col("approved_ts").isNotNull() & (col("approved_ts") < col("purchase_ts"))) |
    (col("carrier_ts").isNotNull() & (col("carrier_ts") < col("purchase_ts"))) |
    (col("customer_ts").isNotNull() & (col("customer_ts") < col("purchase_ts"))) |
    (col("carrier_ts").isNotNull() & col("customer_ts").isNotNull() & (col("customer_ts") < col("carrier_ts")))
).select(
    "order_id",
    "order_status",
    "purchase_ts",
    "approved_ts",
    "carrier_ts",
    "customer_ts"
).show(truncate=False)

In [0]:
# from the above data we are sure about something fishy so we went ahead with counting the no of instances where the above case is true and for that we selected those columns and summed them up to get a total figure of such instances

from pyspark.sql.functions import sum, when

orders_check.select(
    sum(
        when(
            col("approved_ts").isNotNull() &
            (col("approved_ts") < col("purchase_ts")),
            1
        ).otherwise(0)
    ).alias("approved_before_purchase"),

    sum(
        when(
            col("carrier_ts").isNotNull() &
            (col("carrier_ts") < col("purchase_ts")),
            1
        ).otherwise(0)
    ).alias("carrier_before_purchase"),

    sum(
        when(
            col("customer_ts").isNotNull() &
            (col("customer_ts") < col("purchase_ts")),
            1
        ).otherwise(0)
    ).alias("customer_before_purchase"),

    sum(
        when(
            col("carrier_ts").isNotNull() &
            col("customer_ts").isNotNull() &
            (col("customer_ts") < col("carrier_ts")),
            1
        ).otherwise(0)
    ).alias("customer_before_carrier")
).show()

In [0]:
orders_check.filter(
    col("carrier_ts").isNotNull() &
    col("customer_ts").isNotNull() &
    (col("customer_ts") < col("carrier_ts"))
).select(
    "order_id",
    "order_status",
    "purchase_ts",
    "approved_ts",
    "carrier_ts",
    "customer_ts",
    "estimated_ts"
).show(truncate=False)

In [0]:
orders_check.filter(
    col("carrier_ts").isNotNull() &
    (col("carrier_ts") < col("purchase_ts"))
).select(
    "order_id",
    "order_status",
    "purchase_ts",
    "approved_ts",
    "carrier_ts",
    "customer_ts",
    "estimated_ts"
).show(truncate=False)

In [0]:
from pyspark.sql.functions import to_timestamp, to_date, col

orders_check = orders_df.select(
    "*",
    to_timestamp("order_purchase_timestamp").alias("purchase_ts"),
    to_timestamp("order_approved_at").alias("approved_ts"),
    to_timestamp("order_delivered_carrier_date").alias("carrier_ts"),
    to_timestamp("order_delivered_customer_date").alias("customer_ts"),
    to_date("order_estimated_delivery_date").alias("estimated_date")
)

orders_check.select(
    [
        sum(col(c).isNull().cast("int")).alias(c)
        for c in [
            "purchase_ts",
            "approved_ts",
            "carrier_ts",
            "customer_ts",
            "estimated_date"
        ]
    ]
).show()

In [0]:
from pyspark.sql.functions import col, to_timestamp, to_date

# datatype conversion
orders_silver = orders_df.select(
    col("order_id"),
    col("customer_id"),
    col("order_status"),

    to_timestamp("order_purchase_timestamp").alias("order_purchase_timestamp"),
    to_timestamp("order_approved_at").alias("order_approved_at"),
    to_timestamp("order_delivered_carrier_date").alias("order_delivered_carrier_date"),
    to_timestamp("order_delivered_customer_date").alias("order_delivered_customer_date"),
    to_date("order_estimated_delivery_date").alias("order_estimated_delivery_date"),

    col("ingestion_timestamp"),
    col("source_file")
)

In [0]:
# expected schema for orders table and then our function validate_schema being used to compare the current and expected schema

expected_orders_schema = {
    "order_id": "string",
    "customer_id": "string",
    "order_status": "string",
    "order_purchase_timestamp": "timestamp",
    "order_approved_at": "timestamp",
    "order_delivered_carrier_date": "timestamp",
    "order_delivered_customer_date": "timestamp",
    "order_estimated_delivery_date": "date",
    "ingestion_timestamp": "timestamp",
    "source_file": "string"
}

orders_schema_check = validate_schema(
    expected_schema=expected_orders_schema,
    current_schema=orders_silver.schema
)

display(orders_schema_check)

In [0]:
# Validating if there are any null values in the given columns
orders_silver.select([
    sum(col(c).isNull().cast("int")).alias(c)
    for c in [
        "order_id",
        "customer_id",
        "order_status",
        "order_purchase_timestamp",
        "order_estimated_delivery_date"
    ]
]).show()

In [0]:
# write in delta

spark.sql(f"""
CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SILVER_SCHEMA}
""")

(
    orders_silver.write
        .format("delta")
        .mode("overwrite")
        .saveAsTable(f"{CATALOG}.{SILVER_SCHEMA}.orders")
)

In [0]:
# final check if we can get the schema and the table queried
orders_final = spark.table(
    f"{CATALOG}.{SILVER_SCHEMA}.orders"
)

orders_final.printSchema()
display(orders_final.limit(10))

In [0]:
# to check if bronze and silver has same rows this also shows that whether or not we have removed/quarintined some data
print(f"Bronze rows : {orders_df.count():,}")
print(f"Silver rows : {orders_final.count():,}")

In [0]:
# next is where we check the order_items table which has orders and other details and here since one order can have multiple itmes we can have many duplicate for the order_id

order_items_df = spark.table(
    f"{CATALOG}.{BRONZE_SCHEMA}.order_items_raw"
)

profile_table(
    order_items_df,
    table_name="order_items",
    primary_key="order_id"
)

In [0]:
display(order_items_df.limit(10))
# Inspection of the table below by me - each order can have multiple products and so order_id is repeating, order_item_id is basically the item no in the order, say order A has 3 items then we can have this item_id as 1,2,3. Product and seller ids are basically self explainatory, shipping limit is the date by which the seller has to ship the order. Each product can have diff sellers as well and price is product price and freight_value is shipping values

In [0]:
# We are trying to figure out what can be a unique id struct so that querying becommes easier and this can be a combo of order_id and order_item_id as a unique combo and below code tests it

# This now justifys what we thought, both order_id and order_item_id creates a unique pair and now we concretly know that order_item_id is the sequence number of the order 

#order_id = A
#order_item_id = 1,2,3
#order A has 3 items

order_items_df.select(
    "order_id",
    "order_item_id"
).count(), \
order_items_df.select(
    "order_id",
    "order_item_id"
).distinct().count()

In [0]:
# We are trying to count all the nul values in the mentioned columns

order_items_df.select([
    sum(col(c).isNull().cast("int")).alias(c)
    for c in [
        "order_id",
        "order_item_id",
        "product_id",
        "seller_id",
        "shipping_limit_date",
        "price",
        "freight_value"
    ]
]).show()

In [0]:
#we are using describe to check the basic stats for each of the column mentioned
order_items_df.select(
    "order_item_id",
    "price",
    "freight_value"
).describe().show()

In [0]:
#We have to transform the data type of money related columns which are currently in strings and there are two ways one is double and other one is decimal (10,2) which means the data will have max 10 digits out of which 2 will be after decimal point. We are not considering double because it has some complexities when it comes to numbers like 1/3 and all and decimal is stratight forward 10,2 means only 2 digits after decimal and also we can define rounding up and down

from pyspark.sql.functions import col, to_timestamp
order_items_silver = order_items_df.select(
    col("order_id"),
    col("order_item_id").cast("int").alias("order_item_id"),
    col("product_id"),
    col("seller_id"),
    to_timestamp("shipping_limit_date").alias("shipping_limit_date"),
    col("price").cast("decimal(10,2)").alias("price"),
    col("freight_value").cast("decimal(10,2)").alias("freight_value"),
    col("ingestion_timestamp"),
    col("source_file")
)

In [0]:
# now its time to validate the current schema with the expected schema
expected_order_items_schema = {
    "order_id": "string",
    "order_item_id": "int",
    "product_id": "string",
    "seller_id": "string",
    "shipping_limit_date": "timestamp",
    "price": "decimal(10,2)",
    "freight_value": "decimal(10,2)",
    "ingestion_timestamp": "timestamp",
    "source_file": "string"
}

order_items_schema_check = validate_schema(
    expected_schema=expected_order_items_schema,
    current_schema=order_items_silver.schema
)

display(order_items_schema_check)

In [0]:
duplicate_item_count = (
    order_items_silver.count()
    - order_items_silver
        .select("order_id", "order_item_id")
        .distinct()
        .count()
)

print(f"Duplicate (order_id, order_item_id) records: {duplicate_item_count}")

In [0]:
spark.sql(f"""
CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SILVER_SCHEMA}
""")

(
    order_items_silver.write
        .format("delta")
        .mode("overwrite")
        .saveAsTable(f"{CATALOG}.{SILVER_SCHEMA}.order_items")
)

In [0]:
order_items_final = spark.table(
    f"{CATALOG}.{SILVER_SCHEMA}.order_items"
)

order_items_final.printSchema()
display(order_items_final.limit(10))

In [0]:
print(f"Bronze rows : {order_items_df.count():,}")
print(f"Silver rows : {order_items_final.count():,}")

In [0]:
#Now we move onto next table which is products table and we will profile it
products_df = spark.table(
    f"{CATALOG}.{BRONZE_SCHEMA}.products_raw"
)

profile_table(
    products_df,
    table_name="products",
    primary_key="product_id"
)

In [0]:
display(products_df.limit(10))

In [0]:
#We check here if the rows are legitimate null values or random values and form this inspection we found that these are legit null rows all 620, but we have dimensions for them so this can be a glitch so we will keep them as they are not invalid rows

products_df.filter(
    col("product_category_name").isNull()
).select(
    "product_id",
    "product_category_name",
    "product_name_lenght",
    "product_description_lenght",
    "product_photos_qty",
    "product_weight_g",
    "product_length_cm",
    "product_height_cm",
    "product_width_cm"
).show()

In [0]:
#We change the datatype here and also two spell checks on lenght was used for product_name and product_description_lenght and we are changing it to correct spell "length"

products_silver = products_df.select(
    col("product_id"),
    col("product_category_name"),
    col("product_name_lenght").cast("int").alias("product_name_length"),
    col("product_description_lenght").cast("int").alias("product_description_length"),
    col("product_photos_qty").cast("int").alias("product_photos_qty"),
    col("product_weight_g").cast("int").alias("product_weight_g"),
    col("product_length_cm").cast("int").alias("product_length_cm"),
    col("product_height_cm").cast("int").alias("product_height_cm"),
    col("product_width_cm").cast("int").alias("product_width_cm"),
    col("ingestion_timestamp"),
    col("source_file")
)

In [0]:
#Its time to validate schema with expected and current using our own function and for that there are two arguments, expected_schema and current_schema

expected_products_schema = {
    "product_id": "string",
    "product_category_name": "string",
    "product_name_length": "int",
    "product_description_length": "int",
    "product_photos_qty": "int",
    "product_weight_g": "int",
    "product_length_cm": "int",
    "product_height_cm": "int",
    "product_width_cm": "int",
    "ingestion_timestamp": "timestamp",
    "source_file": "string"
}

products_schema_check = validate_schema(
    expected_schema=expected_products_schema,
    current_schema=products_silver.schema
)

display(products_schema_check)

In [0]:
#Since this was all we need to check here we will write the data to the table
spark.sql(f"""
CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SILVER_SCHEMA}
""")

(
    products_silver.write
        .format("delta")
        .mode("overwrite")
        .saveAsTable(f"{CATALOG}.{SILVER_SCHEMA}.products")
)

In [0]:
products_final = spark.table(
    f"{CATALOG}.{SILVER_SCHEMA}.products"
)

products_final.printSchema()
display(products_final.limit(10))

In [0]:
print(f"Bronze rows : {products_df.count():,}")
print(f"Silver rows : {products_final.count():,}")

In [0]:
#We now move with sellers table which is again relatively simple as the products table and we go ahead with again profiling the data and then proceeding accordingly
sellers_df = spark.table(
    f"{CATALOG}.{BRONZE_SCHEMA}.sellers_raw"
)

profile_table(
    sellers_df,
    table_name="sellers",
    primary_key="seller_id"
)

In [0]:
display(sellers_df.limit(10))

In [0]:
#I see we have a number in the seller_city 04482255, hence want to have a check if that is only one case or multiple cases exists
sellers_df.select("seller_city") \
    .distinct() \
    .orderBy("seller_city") \
    .show(10, truncate=False)

In [0]:
#We found 611 unique seller_city and one of them was this integer looking city and so we investigated further and found that there is just one row so it might not be data issue but a real city so we will be keeping it in the table
sellers_df.filter(
    col("seller_city") == "04482255"
).show(truncate=False)

In [0]:
#datatype transformation, we will cast zip_code to integer rest all can be string as it is
from pyspark.sql.functions import col

sellers_silver = sellers_df.select(
    col("seller_id"),
    col("seller_zip_code_prefix").cast("int").alias("seller_zip_code_prefix"),
    col("seller_city"),
    col("seller_state"),
    col("ingestion_timestamp"),
    col("source_file")
)

In [0]:
#schema validation
expected_sellers_schema = {
    "seller_id": "string",
    "seller_zip_code_prefix": "int",
    "seller_city": "string",
    "seller_state": "string",
    "ingestion_timestamp": "timestamp",
    "source_file": "string"
}

sellers_schema_check = validate_schema(
    expected_schema=expected_sellers_schema,
    current_schema=sellers_silver.schema
)

display(sellers_schema_check)

In [0]:
#Write to table
spark.sql(f"""
CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SILVER_SCHEMA}
""")

(
    sellers_silver.write
        .format("delta")
        .mode("overwrite")
        .saveAsTable(f"{CATALOG}.{SILVER_SCHEMA}.sellers")
)

In [0]:
sellers_final = spark.table(
    f"{CATALOG}.{SILVER_SCHEMA}.sellers"
)

sellers_final.printSchema()
display(sellers_final.limit(10))

In [0]:
print(f"Bronze rows : {sellers_df.count():,}")
print(f"Silver rows : {sellers_final.count():,}")

In [0]:
#Payments table profiling
payments_df = spark.table(
    f"{CATALOG}.{BRONZE_SCHEMA}.payments_raw"
)

profile_table(
    payments_df,
    table_name="payments",
    primary_key="order_id"
)

In [0]:
display(payments_df.limit(10))

In [0]:
payments_df.filter(
    col("payment_sequential") == "3"
).show(truncate=False)

In [0]:
#Installments is a overall view of the no of installments the customer chose to pay that amount, and payment sequential is something weird which shows that each order can have multiple payments one direct one with voucher and so on so each order say order_a can have 2 sequenctial payments like 1 and 2. Here we are trying to check if order_id and sequential_id makes a unique pair and if yes then we found the composite key for this as well
#Results shows duplicate record as 0, that means this will be a good composite primary key for us
payment_key_check = (
    payments_df.count()
    - payments_df
        .select("order_id", "payment_sequential")
        .distinct()
        .count()
)

print(f"Duplicate (order_id, payment_sequential): {payment_key_check}")

In [0]:
#Once we saw that this combo works we also need to check how many orders have how many payment sequence to get a clear picture of what exactly is happening under the hood.
#We see based on the result that there are 96k records with 1 payment sequence but we also see for some orders there are more than 20+ payment sequence
#96,479 orders with exactly 1 payment record
payments_df.groupBy("order_id") \
    .count() \
    .withColumnRenamed("count", "payment_records_per_order") \
    .groupBy("payment_records_per_order") \
    .count() \
    .withColumnRenamed("count", "number_of_orders") \
    .orderBy("payment_records_per_order") \
    .show()

In [0]:
payments_df.groupBy("order_id") \
    .count() \
    .orderBy(col("count").desc()) \
    .limit(1) \
    .show()

In [0]:
max_payment_order = (
    payments_df.groupBy("order_id")
    .count()
    .orderBy(col("count").desc())
    .first()["order_id"]
)

payments_df.filter(
    col("order_id") == max_payment_order
).orderBy(
    col("payment_sequential")
).show(truncate=False)

In [0]:
#Check the composite keys
payment_key_check = (
    payments_df.count()
    - payments_df
        .select("order_id", "payment_sequential")
        .distinct()
        .count()
)

print(f"Duplicate (order_id, payment_sequential): {payment_key_check}")

In [0]:
#We now add some details on what we exactly require on this specifically on the data type and we cast it accordingly
from pyspark.sql.functions import col

payments_silver = payments_df.select(
    col("order_id"),
    col("payment_sequential").cast("int").alias("payment_sequential"),
    col("payment_type"),
    col("payment_installments").cast("int").alias("payment_installments"),
    col("payment_value").cast("decimal(10,2)").alias("payment_value"),
    col("ingestion_timestamp"),
    col("source_file")
)

In [0]:
#validate schema
expected_payments_schema = {
    "order_id": "string",
    "payment_sequential": "int",
    "payment_type": "string",
    "payment_installments": "int",
    "payment_value": "decimal(10,2)",
    "ingestion_timestamp": "timestamp",
    "source_file": "string"
}

payments_schema_check = validate_schema(
    expected_schema=expected_payments_schema,
    current_schema=payments_silver.schema
)

display(payments_schema_check)

In [0]:
spark.sql(f"""
CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SILVER_SCHEMA}
""")

(
    payments_silver.write
        .format("delta")
        .mode("overwrite")
        .saveAsTable(f"{CATALOG}.{SILVER_SCHEMA}.payments")
)

In [0]:
payments_final = spark.table(
    f"{CATALOG}.{SILVER_SCHEMA}.payments"
)

payments_final.printSchema()
display(payments_final.limit(10))

In [0]:
print(f"Bronze rows : {payments_df.count():,}")
print(f"Silver rows : {payments_final.count():,}")

In [0]:
#To be honest we can make a function for write table which also validates it shows the rows from bronze and silver but that fine now
#Next three tables are relatively easy and not that required so lets fasten up
reviews_df = spark.table(
    f"{CATALOG}.{BRONZE_SCHEMA}.reviews_raw"
)

profile_table(
    reviews_df,
    table_name="reviews",
    primary_key="review_id"
)

In [0]:
print("Distinct order_id:", reviews_df.select("order_id").distinct().count())
print("Distinct review_id:", reviews_df.select("review_id").distinct().count())

print("\nDuplicate review_id examples:")
reviews_df.groupBy("review_id") \
    .count() \
    .filter(col("count") > 1) \
    .orderBy(col("count").desc()) \
    .show(20, truncate=False)

In [0]:
reviews_df.filter(
    col("review_id") == "Recomendo"
).show(truncate=False)

In [0]:
reviews_df.select(
    "review_id",
    "order_id",
    "review_score",
    "review_comment_title",
    "review_comment_message",
    "review_creation_date",
    "review_answer_timestamp"
).show(20, truncate=False)

In [0]:
reviews_df = spark.table(
    f"{CATALOG}.{BRONZE_SCHEMA}.reviews_raw"
)

In [0]:
profile_table(
    reviews_df,
    table_name="reviews",
    primary_key="review_id"
)

In [0]:
from pyspark.sql.functions import col

invalid_review_id_count = reviews_df.filter(
    ~col("review_id").rlike("^[0-9a-f]{32}$")
).count()

print(f"Invalid review_id records: {invalid_review_id_count}")

In [0]:
reviews_df.filter(
    ~col("review_id").rlike("^[0-9a-f]{32}$")
).select(
    "review_id",
    "order_id",
    "review_score",
    "review_comment_title",
    "review_comment_message",
    "review_creation_date",
    "review_answer_timestamp"
).show(20, truncate=False)

In [0]:
valid_reviews_df = reviews_df.filter(
    col("review_id").rlike("^[0-9a-f]{32}$")
)

print(f"Bronze rows          : {reviews_df.count():,}")
print(f"Invalid review rows  : {reviews_df.count() - valid_reviews_df.count():,}")
print(f"Valid review rows    : {valid_reviews_df.count():,}")

In [0]:
reviews_silver = valid_reviews_df.select(
    col("review_id"),
    col("order_id"),
    col("review_score").cast("int").alias("review_score"),
    col("review_comment_title"),
    col("review_comment_message"),
    to_timestamp("review_creation_date").alias("review_creation_date"),
    to_timestamp("review_answer_timestamp").alias("review_answer_timestamp"),
    col("ingestion_timestamp"),
    col("source_file")
)

In [0]:
# Data Quality Note:
# During Bronze validation, the reviews dataset contained 31 malformed records
# where review_id did not follow the expected 32-character hexadecimal ID format.
# Investigation showed that these records were caused by malformed CSV parsing,
# with review text and timestamp values shifted into incorrect columns.
#
# Decision:
# These records are retained in Bronze for source traceability but excluded from
# Silver rather than attempting to reconstruct them. Silver will contain only
# records with a valid review_id, while the 31 excluded records are treated as
# rejected/malformed data and should be captured in the project's data-quality
# documentation.

In [0]:
reviews_silver = valid_reviews_df.select(
    col("review_id"),
    col("order_id"),
    col("review_score").cast("int").alias("review_score"),
    col("review_comment_title"),
    col("review_comment_message"),
    to_timestamp("review_creation_date").alias("review_creation_date"),
    to_timestamp("review_answer_timestamp").alias("review_answer_timestamp"),
    col("ingestion_timestamp"),
    col("source_file")
)

In [0]:
expected_reviews_schema = {
    "review_id": "string",
    "order_id": "string",
    "review_score": "int",
    "review_comment_title": "string",
    "review_comment_message": "string",
    "review_creation_date": "timestamp",
    "review_answer_timestamp": "timestamp",
    "ingestion_timestamp": "timestamp",
    "source_file": "string"
}

reviews_schema_check = validate_schema(
    expected_schema=expected_reviews_schema,
    current_schema=reviews_silver.schema
)

display(reviews_schema_check)

In [0]:
duplicate_review_count = (
    reviews_silver.count()
    - reviews_silver.select("review_id").distinct().count()
)

print(f"Duplicate review_id records: {duplicate_review_count}")

In [0]:
(
    reviews_silver.write
        .format("delta")
        .mode("overwrite")
        .saveAsTable(
            f"{CATALOG}.{SILVER_SCHEMA}.reviews"
        )
)

In [0]:
#We tried to write it but got same error where we have random characters in the timestamp as they are string so these are caused by certain type of commas and semi-colon so we have to remove them because that might casue issues going forward

from pyspark.sql.functions import expr

reviews_checked = reviews_df.withColumn(
    "creation_ts_check",
    expr("try_cast(review_creation_date AS TIMESTAMP)")
).withColumn(
    "answer_ts_check",
    expr("try_cast(review_answer_timestamp AS TIMESTAMP)")
)

reviews_checked.filter(
    col("review_creation_date").isNotNull() &
    col("creation_ts_check").isNull()
).count()

In [0]:
valid_reviews_df = reviews_checked.filter(
    col("review_id").rlike("^[0-9a-f]{32}$") &
    col("order_id").rlike("^[0-9a-f]{32}$") &
    col("review_score").rlike("^[1-5]$") &
    (
        col("review_creation_date").isNull() |
        col("creation_ts_check").isNotNull()
    ) &
    (
        col("review_answer_timestamp").isNull() |
        col("answer_ts_check").isNotNull()
    )
)

In [0]:
print(f"Bronze rows : {reviews_df.count():,}")
print(f"Valid rows  : {valid_reviews_df.count():,}")
print(f"Rejected    : {reviews_df.count() - valid_reviews_df.count():,}")

In [0]:
reviews_silver = valid_reviews_df.select(
    col("review_id"),
    col("order_id"),
    col("review_score").cast("int").alias("review_score"),
    col("review_comment_title"),
    col("review_comment_message"),
    col("creation_ts_check").alias("review_creation_date"),
    col("answer_ts_check").alias("review_answer_timestamp"),
    col("ingestion_timestamp"),
    col("source_file")
)

In [0]:
from pyspark.sql.functions import col, expr

reviews_checked = (
    reviews_df
    .withColumn(
        "creation_ts_check",
        expr("try_cast(review_creation_date AS TIMESTAMP)")
    )
    .withColumn(
        "answer_ts_check",
        expr("try_cast(review_answer_timestamp AS TIMESTAMP)")
    )
)

valid_reviews_df = reviews_checked.filter(
    col("review_id").rlike("^[0-9a-f]{32}$") &
    col("order_id").rlike("^[0-9a-f]{32}$") &
    col("review_score").rlike("^[1-5]$") &
    (
        col("review_creation_date").isNull() |
        col("creation_ts_check").isNotNull()
    ) &
    (
        col("review_answer_timestamp").isNull() |
        col("answer_ts_check").isNotNull()
    )
)

reviews_silver = valid_reviews_df.select(
    col("review_id"),
    col("order_id"),
    col("review_score").cast("int").alias("review_score"),
    col("review_comment_title"),
    col("review_comment_message"),
    col("creation_ts_check").alias("review_creation_date"),
    col("answer_ts_check").alias("review_answer_timestamp"),
    col("ingestion_timestamp"),
    col("source_file")
)

In [0]:
expected_reviews_schema = {
    "review_id": "string",
    "order_id": "string",
    "review_score": "int",
    "review_comment_title": "string",
    "review_comment_message": "string",
    "review_creation_date": "timestamp",
    "review_answer_timestamp": "timestamp",
    "ingestion_timestamp": "timestamp",
    "source_file": "string"
}

reviews_schema_check = validate_schema(
    expected_schema=expected_reviews_schema,
    current_schema=reviews_silver.schema
)

display(reviews_schema_check)

In [0]:
(
    reviews_silver.write
        .format("delta")
        .mode("overwrite")
        .saveAsTable(
            f"{CATALOG}.{SILVER_SCHEMA}.reviews"
        )
)

reviews_final = spark.table(
    f"{CATALOG}.{SILVER_SCHEMA}.reviews"
)

reviews_final.printSchema()

In [0]:
print(f"Bronze rows   : {reviews_df.count():,}")
print(f"Silver rows   : {reviews_final.count():,}")

In [0]:
#Now the last two tables and one of them is Geolocation, as usual we will profile it and then inspect it
geolocation_df = spark.table(
    f"{CATALOG}.{BRONZE_SCHEMA}.geolocation_raw"
)

profile_table(
    geolocation_df,
    table_name="geolocation",
    primary_key="geolocation_zip_code_prefix"
)

In [0]:
#There is not much here, we do not have a primary key as this is just a info table and the data type will be mostly sting or int or double
from pyspark.sql.functions import col

geolocation_silver = geolocation_df.select(
    col("geolocation_zip_code_prefix").cast("int").alias(
        "geolocation_zip_code_prefix"
    ),
    col("geolocation_lat").cast("double").alias("geolocation_lat"),
    col("geolocation_lng").cast("double").alias("geolocation_lng"),
    col("geolocation_city"),
    col("geolocation_state"),
    col("ingestion_timestamp"),
    col("source_file")
)

In [0]:
expected_geolocation_schema = {
    "geolocation_zip_code_prefix": "int",
    "geolocation_lat": "double",
    "geolocation_lng": "double",
    "geolocation_city": "string",
    "geolocation_state": "string",
    "ingestion_timestamp": "timestamp",
    "source_file": "string"
}

geolocation_schema_check = validate_schema(
    expected_schema=expected_geolocation_schema,
    current_schema=geolocation_silver.schema
)

display(geolocation_schema_check)

In [0]:
(
    geolocation_silver.write
        .format("delta")
        .mode("overwrite")
        .saveAsTable(
            f"{CATALOG}.{SILVER_SCHEMA}.geolocation"
        )
)

geolocation_final = spark.table(
    f"{CATALOG}.{SILVER_SCHEMA}.geolocation"
)

print(f"Bronze rows : {geolocation_df.count():,}")
print(f"Silver rows : {geolocation_final.count():,}")

In [0]:
#Final table called Product Category table
spark.sql(f"SHOW TABLES IN {CATALOG}.{BRONZE_SCHEMA}").show(
    truncate=False
)

In [0]:
category_translation_df = spark.table(
    f"{CATALOG}.{BRONZE_SCHEMA}.product_category_translation_raw"
)

In [0]:
category_translation_df = spark.table(
    f"{CATALOG}.{BRONZE_SCHEMA}.product_category_translation_raw"
)

In [0]:
profile_table(
    category_translation_df,
    table_name="product_category_translation",
    primary_key="product_category_name"
)

In [0]:
category_translation_silver = category_translation_df.select(
    col("product_category_name"),
    col("product_category_name_english"),
    col("ingestion_timestamp"),
    col("source_file")
)

In [0]:
expected_category_translation_schema = {
    "product_category_name": "string",
    "product_category_name_english": "string",
    "ingestion_timestamp": "timestamp",
    "source_file": "string"
}

category_translation_schema_check = validate_schema(
    expected_schema=expected_category_translation_schema,
    current_schema=category_translation_silver.schema
)

display(category_translation_schema_check)

In [0]:
(
    category_translation_silver.write
        .format("delta")
        .mode("overwrite")
        .saveAsTable(
            f"{CATALOG}.{SILVER_SCHEMA}.product_category_translation"
        )
)

In [0]:
category_translation_final = spark.table(
    f"{CATALOG}.{SILVER_SCHEMA}.product_category_translation"
)

print(f"Bronze rows : {category_translation_df.count():,}")
print(f"Silver rows : {category_translation_final.count():,}")

In [0]:
#Final Checks on the Silver Table before we commit to Github
silver_tables = [
    "customers",
    "orders",
    "order_items",
    "products",
    "sellers",
    "payments",
    "reviews",
    "geolocation",
    "product_category_translation"
]

for table in silver_tables:
    df = spark.table(f"{CATALOG}.{SILVER_SCHEMA}.{table}")
    
    print(
        f"{table:<35} "
        f"rows={df.count():>8,} "
        f"columns={len(df.columns):>2}"
    )

In [0]:
spark.sql(
    f"SHOW TABLES IN {CATALOG}.{SILVER_SCHEMA}"
).show(truncate=False)